# Data Collection & Indicator calucation

In [91]:
import numpy as np
import yfinance as yf
import pandas as pd

def get_data(ticker, period, interval='1d'):
    data = yf.download(ticker, period=period, interval=interval)
    data.columns= ["Close",	"High",	"Low",	"Open", "Volume"]
    return data

ticker = 'AAPL'

## Performance Testing Function:

In [92]:
def calculate_strategy_returns( predictions, data_set ):
    returns = data_set["Close"].pct_change()[-len(predictions):]

    predictions = predictions.astype(float)
    predictions[ predictions == 0 ] = -1

    strategy_returns = predictions * returns
    cumulative_returns = np.cumsum(strategy_returns)
    return cumulative_returns[-1]

def calculate_strategy_rate(model_returns, data_set, test_set):
    total_returns = data_set["Close"].pct_change()[-len(test_set)-1:].dropna().to_numpy()
    total_returns[ total_returns < 0 ] *= -1
    return model_returns / np.sum(total_returns)

In [93]:
indicators_groups ={
    # Trend Indicators
    "trend_indicators" : ['SMA20', 'EMA20', 'WMA20', 'SAR', 'CCI'],

    "trend_indicators_1" : ['SMA20', 'EMA20', 'SAR'],
    
    # Momentum Indicators
    "momentum_indicators" : ['RSI', 'MACD', 'MACD_Signal', 'MACD_Hist', 'Stoch_K', 'Stoch_D', 'ADX'],

    # Volatility Indicators
    "volatility_indicators" : ['BB_High', 'BB_Low', 'ATR'],

    # Volume Indicators
    "volume_indicators" : [ 'OBV', 'CMF', 'VWAP'],

    # Composite Indicators
    "composite_indicators" : ['Ichimoku_Base', 'Ichimoku_Conversion']
}

# Signal-Based Strategies

## Data Preparation:

In [94]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def data_test_set(data):
    scaler = StandardScaler()
    X= pd.DataFrame(data= scaler.fit_transform(data.copy()), columns= data.columns)
    y= np.where(data["Close"].shift(-1) > data["Close"], 1, 0)

    return X, y

### Indicators

In [95]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error

# Signal Generators

def sma_signal(data, period=20):
    sma = data['Close'].rolling(window=period).mean()
    buy = data['Close'] > sma
    sell = data['Close'] < sma
    return buy, sell


def ema_signal(data, period=20):
    ema = data['Close'].ewm(span=period, adjust=False).mean()
    buy = data['Close'] > ema
    sell = data['Close'] < ema
    return buy, sell


def rsi_signal(data, period=14, overbought=70, oversold=30):
    delta = data['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    buy = rsi < oversold
    sell = rsi > overbought
    return buy, sell


def macd_signal(data, fast=12, slow=26, signal=9):
    fast_ema = data['Close'].ewm(span=fast, adjust=False).mean()
    slow_ema = data['Close'].ewm(span=slow, adjust=False).mean()
    macd_line = fast_ema - slow_ema
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    buy = macd_line > signal_line
    sell = macd_line < signal_line
    return buy, sell


def bollinger_signal(data, period=20):
    sma = data['Close'].rolling(window=period).mean()
    std_dev = data['Close'].rolling(window=period).std()
    upper_band = sma + (2 * std_dev)
    lower_band = sma - (2 * std_dev)
    buy = data['Close'] < lower_band
    sell = data['Close'] > upper_band
    return buy, sell


def adx_signal(data, period=14):
    high_diff = data['High'].diff()
    low_diff = data['Low'].diff()
    plus_dm = np.where((high_diff > low_diff) & (high_diff > 0), high_diff, 0)
    minus_dm = np.where((low_diff > high_diff) & (low_diff > 0), low_diff, 0)
    tr = np.maximum(data['High'] - data['Low'],
                   np.maximum(abs(data['High'] - data['Close'].shift(1)),
                              abs(data['Low'] - data['Close'].shift(1))))
    atr = tr.rolling(window=period).mean()
    plus_di = 100 * (np.roll(plus_dm, shift=period).mean() / atr)
    minus_di = 100 * (np.roll(minus_dm, shift=period).mean() / atr)
    dx = abs(plus_di - minus_di) / (plus_di + minus_di) * 100
    adx = dx.rolling(window=period).mean()
    buy = adx > 25
    sell = adx < 20
    return buy, sell

def stochastic_rsi_signal(data, period=14):
    rsi = rsi_signal(data)[0].astype(float)
    stoch_rsi = (rsi - rsi.rolling(window=period).min()) / (rsi.rolling(window=period).max() - rsi.rolling(window=period).min())
    buy = stoch_rsi < 0.2
    sell = stoch_rsi > 0.8
    return buy, sell


def macd_signal(data, fast=12, slow=26, signal=9):
    fast_ema = data['Close'].ewm(span=fast, adjust=False).mean()
    slow_ema = data['Close'].ewm(span=slow, adjust=False).mean()
    macd_line = fast_ema - slow_ema
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    buy = macd_line > signal_line
    sell = macd_line < signal_line
    return buy, sell


def williams_r_signal(data, period=14):
    highest_high = data['High'].rolling(window=period).max()
    lowest_low = data['Low'].rolling(window=period).min()
    williams_r = -100 * (highest_high - data['Close']) / (highest_high - lowest_low)
    buy = williams_r < -80
    sell = williams_r > -20
    return buy, sell


def atr_signal(data, period=14):
    high_low = data['High'] - data['Low']
    high_close = np.abs(data['High'] - data['Close'].shift())
    low_close = np.abs(data['Low'] - data['Close'].shift())
    tr = high_low.combine(high_close, max).combine(low_close, max)
    atr = tr.rolling(window=period).mean()
    buy = atr > atr.shift()
    sell = atr < atr.shift()
    return buy, sell


def sar_signal(data, af=0.02, max_af=0.2):
    sar = data['Close'].shift()
    trend = 1
    af_value = af
    buy = pd.Series(False, index=data.index)
    sell = pd.Series(False, index=data.index)
    for i in range(1, len(data)):
        if trend == 1:
            sar[i] = sar[i - 1] + af_value * (data['High'][i - 1] - sar[i - 1])
            if data['Close'][i] < sar[i]:
                trend = -1
                sar[i] = max(data['High'][i], data['High'][i - 1])
                sell[i] = True
        else:
            sar[i] = sar[i - 1] + af_value * (data['Low'][i - 1] - sar[i - 1])
            if data['Close'][i] > sar[i]:
                trend = 1
                sar[i] = min(data['Low'][i], data['Low'][i - 1])
                buy[i] = True
        af_value = min(af_value + af, max_af)
    return buy, sell


# Combine Signals
def aggregate_signals(buy_signals, sell_signals):
    buy = np.any(np.array(buy_signals), axis=0)
    sell = np.any(np.array(sell_signals), axis=0)
    return buy, sell

In [96]:
indicators_groups_1 ={
    # Trend Indicators
    "trend_indicators" : ["sma_signal", "ema_signal", "sar_signal"],

    # Momentum Indicators
    "momentum_indicators" : ["macd_signal", "rsi_signal", "adx_signal", "stochastic_rsi_signal", "williams_r_signal"],

    # Volatility Indicators
    "volatility_indicators" : ["atr_signal", "bollinger_signal"],
}

signals_groups = {
    "sma_signal": sma_signal,
    "ema_signal": ema_signal,
    "sar_signal": sar_signal,
    "macd_signal": macd_signal,
    "rsi_signal": rsi_signal,
    "adx_signal": adx_signal,
    "stochastic_rsi_signal": stochastic_rsi_signal,
    "williams_r_signal": williams_r_signal,
    "atr_signal": atr_signal,
    "bollinger_signal": bollinger_signal
}

## Performance Testing

In [97]:
# Generate signals for each indicator
def signal_strategy( data, model_set ):
    buy_signals = []
    sell_signals = []

    for signal_func in model_set:
        buy, sell = signal_func(data)
        buy_signals.append(buy)
        sell_signals.append(sell)
    return buy_signals, sell_signals

In [98]:
from sklearn.model_selection import TimeSeriesSplit

def singal_strategy_trading( data ):
    result = 0   
    
    X, y = data_test_set(data= data)
    
    for group, strategy in indicators_groups_1.items():
        accuracy, f1, rmse, rate_of_return, sharpe = np.array([]), np.array([]), np.array([]), np.array([]), np.array([])
        
        for train_idx, test_idx in TimeSeriesSplit(n_splits=5).split( X.copy() ):
            # Split the data into training and testing sets based on time order
            X_train_2, X_test_2 = X.iloc[train_idx], X.iloc[test_idx]
            y_train_2, y_test_2 = y[train_idx[0]:train_idx[-1]+1], y[test_idx[0]: test_idx[-1]+1]

            buy_signals, sell_signals = signal_strategy( X_test_2.copy().reset_index(), [signals_groups.get( x ) for x in strategy] )

            # Aggregate signals
            final_buy, final_sell = aggregate_signals( buy_signals, sell_signals )

            signals= final_buy.copy().astype(int)
            
            ### performance metrics test
            accuracy= np.append( accuracy, accuracy_score(y_test_2, signals) )
            f1= np.append( f1, f1_score(y_test_2, signals) )
            rmse= np.append( rmse, mean_squared_error(y_test_2, signals) )

            ### Cummulative Return and Balance Test:
            returns = data.copy().iloc[test_idx[0]-1: test_idx[-1]+1]["Close"].copy().pct_change().dropna().to_numpy()
            
            total_return = returns.copy()
            total_return[ total_return < 0 ] *= -1
            total_return= total_return.sum()
            
            signals[ signals == 0 ] = -1
            
            cumulative_returns = np.sum(returns * signals)
            sharpe = np.append( sharpe, np.mean(returns * signals)/np.std(returns * signals) )
            rate_of_return = np.append( rate_of_return, cumulative_returns / total_return )
            rate_of_return = np.mean(rate_of_return)

        ### Returns Test
        if rate_of_return > result:
            result = rate_of_return
            
    return result

## Optimal year for signal strategy:

In [99]:
from bayes_opt import BayesianOptimization

def evaluate_signal_strategy(years):
    years = str(int(years)) + "y"
    data= get_data(ticker= ticker, period= years)
    return singal_strategy_trading(data = data.copy())

def optimize_years():
    optimizer = BayesianOptimization(
        f= evaluate_signal_strategy,
        pbounds={'years': (3, 10)},
        random_state=42
    )
    optimizer.maximize( init_points= 5, n_iter= 10 )
    return optimizer.max

best_years = optimize_years()

|   iter    |  target   |   years   |
-------------------------------------


[*********************100%***********************]  1 of 1 completed


| 1         | 0.4046    | 5.622     |


[*********************100%***********************]  1 of 1 completed


| 2         | 0.3959    | 9.655     |


...

| 13        | 0.393     | 6.307     |


[*********************100%***********************]  1 of 1 completed


| 14        | 0.4194    | 3.399     |


[*********************100%***********************]  1 of 1 completed

| 15        | 0.4194    | 3.064     |


In [100]:
print(f"Best training period: {best_years}")

Best training period: {'target': 0.41941078752759997, 'params': {'years': 3.0003368672601822}}


# Model-Based Strategies

## Tesing without hyperparameters tunning

In [101]:
## feature_selection.py
from sklearn.decomposition import PCA

# Apply PCA
def pca(n_components, X_train, indicators_groups):
    pca = PCA(n_components= n_components)  # Keep top 5 components
    X_train_pca = pca.fit_transform(X_train)

    # Check how much variance is retained
    pca_components = pd.DataFrame(pca.components_, columns= X_train.columns, index=[f'PC{i+1}' for i in range(pca.n_components_)])

    # Identify the Most Important Features in Each Principal Component
    top_features_per_pc = pca_components.abs().idxmax(axis=1)
    unique_selected_features = list(set(top_features_per_pc))

    # Print Selected Features
    # print("Selected Features After PCA Reduction:", unique_selected_features)

    indicators_groups["Custom Set"] = unique_selected_features

In [109]:
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC

import ta
import pandas_ta as pta

class model_strategy_trading():
    def __init__(self, data):
        self.data = data
        self.indicators_groups = indicators_groups_1
        self.X, self.y = self.data_signal_preparation()
    
    def data_value_preparation(self):

        data = self.data.copy()
        indicators = pd.DataFrame(index= data.index)
        
        # Moving Averages
        indicators['SMA20'] = ta.trend.sma_indicator(data['Close'], window=20)
        indicators['EMA20'] = ta.trend.ema_indicator(data['Close'], window=20)
        indicators['WMA20'] = ta.trend.wma_indicator(data['Close'], window=20)

        # Relative Strength Index (RSI)
        indicators['RSI'] = ta.momentum.rsi(data['Close'], window=14)

        # MACD
        indicators['MACD'] = ta.trend.macd(data['Close'])
        indicators['MACD_Signal'] = ta.trend.macd_signal(data['Close'])
        indicators['MACD_Hist'] = ta.trend.macd_diff(data['Close'])

        # Bollinger Bands
        bb = ta.volatility.BollingerBands(data['Close'], window=20)
        indicators['BB_High'] = bb.bollinger_hband()
        indicators['BB_Low'] = bb.bollinger_lband()

        # Average True Range (ATR)
        indicators['ATR'] = ta.volatility.average_true_range(data['High'], data['Low'], data['Close'], window=14)

        # Stochastic Oscillator
        indicators['Stoch_K'] = ta.momentum.stoch(data['High'], data['Low'], data['Close'], window=14)
        indicators['Stoch_D'] = ta.momentum.stoch_signal(data['High'], data['Low'], data['Close'], window=14)

        # On-Balance Volume (OBV)
        indicators['OBV'] = ta.volume.on_balance_volume(data['Close'], data['Volume'])

        # Chaikin Money Flow (CMF)
        indicators['CMF'] = ta.volume.chaikin_money_flow(data['High'], data['Low'], data['Close'], data['Volume'], window=20)

        # Commodity Channel Index (CCI)
        indicators['CCI'] = ta.trend.cci(data['High'], data['Low'], data['Close'], window=20)

        # Parabolic SAR
        psar = pta.psar(data['High'], data['Low'], data['Close'])
        psar_col = [col for col in psar.columns if 'PSAR' in col][0]  # Find the PSAR column
        indicators['SAR'] = psar[psar_col]

        # Average Directional Index (ADX)
        indicators['ADX'] = ta.trend.adx(data['High'], data['Low'], data['Close'], window=14)

        # Ichimoku Cloud
        indicators['Ichimoku_Base'] = ta.trend.ichimoku_base_line(data['High'], data['Low'])
        indicators['Ichimoku_Conversion'] = ta.trend.ichimoku_conversion_line(data['High'], data['Low'])

        # Volume Weighted Average Price (VWAP)
        indicators['VWAP'] = ta.volume.volume_weighted_average_price(data['High'], data['Low'], data['Close'], data['Volume'])

        X = indicators.copy().dropna()
        y= np.where( self.data["Close"].shift(-1) > self.data["Close"], 1, 0)[-len(X):]
        
        X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, shuffle=False)
        
        return X_train, X_test, y_train, y_test
    
    def data_signal_preparation(self):
        buy_signals, sell_signals = signal_strategy( self.data.copy(), signals_groups.values() )
        data_2 = np.array( [ x.to_numpy() for x in buy_signals] ).astype(int).T
        
        X = pd.DataFrame( data= data_2, columns= signals_groups.keys() )
        y= np.where( self.data["Close"].shift(-1) > self.data["Close"], 1, 0)
        
        return X, y
    
    def run(self, model):
        X, y = self.X.copy(), self.y.copy()
        
        for n in range(2, len(X.columns)):
            pca(n_components= n, X_train= X.copy(), indicators_groups= self.indicators_groups)
    
        last_sharpe, last_f1, last_rate_of_return, last_rmse, accuracy = 0, 0, 0, 0 ,0
        for group in self.indicators_groups:
            
            X_1 = X.copy().loc[:, self.indicators_groups[group]]
            
            for name, model in models.items():
                accur, f1, rmse, rate_of_return, sharpe = np.array([]), np.array([]), np.array([]), np.array([]), np.array([])
                
                for train_idx, test_idx in TimeSeriesSplit(n_splits=5).split( X_1.copy() ):
                    # Split the data into training and testing sets based on time order
                    X_train_2, X_test_2 = X_1.iloc[train_idx], X_1.iloc[test_idx]
                    y_train_2, y_test_2 = y[train_idx[0]:train_idx[-1]+1], y[test_idx[0]: test_idx[-1]+1]

                    model.fit(X_train_2, y_train_2)
                    y_pred = model.predict(X_test_2)

                    if len(np.unique(y_pred)) == 1:
                        continue
                    
                    accur= np.append( accur, accuracy_score(y_test_2, y_pred) )
                    f1 = np.append( f1, f1_score(y_test_2, y_pred) )
                    rmse = np.append( rmse, mean_squared_error(y_test_2, y_pred) )

                    ### Cummulative Return and Balance Test:
                    returns = self.data.copy().reset_index().iloc[test_idx[0]-1: test_idx[-1]+1]["Close"].copy().pct_change().dropna().to_numpy()
                    total_return = returns.copy()
                    total_return[ total_return < 0 ] *= -1
                    total_return= np.sum( total_return )
                    
                    y_pred[ y_pred == 0 ] = -1
                    
                    cumulative_returns = np.sum(returns * y_pred)
                    rate_of_return = np.append( rate_of_return, cumulative_returns / total_return )
                    sharpe = np.append( sharpe, np.mean(returns * y_pred)/np.std(returns * y_pred) )
                    
                accur, f1, rmse, rate_of_return, sharpe = (np.mean(i) for i in [accur, f1, rmse, rate_of_return, sharpe])
            
            #### Record for the final result.
            if sharpe > last_sharpe:
                last_sharpe = sharpe
                last_rate_of_return = rate_of_return
                accuracy = accur
                last_f1 = f1
                last_rmse = rmse
                feature_set = group
                method = name

            elif last_sharpe == sharpe:
                if rate_of_return > last_rate_of_return:
                    last_rate_of_return = rate_of_return
                    accuracy = accur
                    last_f1 = f1
                    last_rmse = rmse
                    feature_set = group
                    method = name

                elif rate_of_return == last_rate_of_return:            
                    if accuracy < accur:
                        accuracy = accur
                        last_f1 = f1
                        last_rmse = rmse
                        feature_set = group
                        method = name

                    elif accuracy == accur:
                        if last_f1 < f1:
                            last_f1 = f1
                            last_rmse = rmse
                            feature_set = group
                            method = name

                        elif last_f1 == f1:
                            if rmse > rmse:
                                last_rmse = rmse
                                feature_set = group
                                method = name
        
        return last_rate_of_return

## Optimal year for model strategy

In [ ]:
def evaluate_model_strategy(years):
    years = str(int(years)) + "y"
    data= get_data(ticker= ticker, period= years)
    return model_strategy_trading(data= data.copy()).run(model= model)

def optimize_years():
    optimizer = BayesianOptimization(
        f= evaluate_model_strategy,
        pbounds={'years': (3, 10)},
        random_state=42
    )
    optimizer.maximize( init_points=5, n_iter=10 )
    return optimizer.max

models = {
            'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
            'Logistic Regression': LogisticRegression(),
            'SVM': SVC(),
            'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss')
        }
best_years = []
for model_name, the_model in models.items():
    model = the_model
    best_years.append( optimize_years() )

|   iter    |  target   |   years   |
-------------------------------------


[*********************100%***********************]  1 of 1 completed
/var/folders/j_/0q2qhjf90s7gbt6hw8zb3_jc0000gn/T/ipykernel_44168/886000332.py:117: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  sar[i] = sar[i - 1] + af_value * (data['High'][i - 1] - sar[i - 1])
/var/folders/j_/0q2qhjf90s7gbt6hw8zb3_jc0000gn/T/ipykernel_44168/886000332.py:117: FutureWarning: Series.__setitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To set a value by position, use `ser.iloc[pos] = value`
  sar[i] = sar[i - 1] + af_value * (data['High'][i - 1] - sar[i - 1])
/var/folders/j_/0q2qhjf90s7gbt6hw8zb3_jc0000gn/T/ipykernel_44168/886000332.py:118: FutureWarning: Series.__getitem__ treating keys as positions is deprecate

| 1         | 0.1997    | 5.622     |


[*********************100%***********************]  1 of 1 completed
/var/folders/j_/0q2qhjf90s7gbt6hw8zb3_jc0000gn/T/ipykernel_44168/886000332.py:117: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  sar[i] = sar[i - 1] + af_value * (data['High'][i - 1] - sar[i - 1])
/var/folders/j_/0q2qhjf90s7gbt6hw8zb3_jc0000gn/T/ipykernel_44168/886000332.py:117: FutureWarning: Series.__setitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To set a value by position, use `ser.iloc[pos] = value`
  sar[i] = sar[i - 1] + af_value * (data['High'][i - 1] - sar[i - 1])
/var/folders/j_/0q2qhjf90s7gbt6hw8zb3_jc0000gn/T/ipykernel_44168/886000332.py:118: FutureWarning: Series.__getitem__ treating keys as positions is deprecate

| 2         | 0.2836    | 9.655     |


...

[*********************100%***********************]  1 of 1 completed
/var/folders/j_/0q2qhjf90s7gbt6hw8zb3_jc0000gn/T/ipykernel_44168/886000332.py:117: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  sar[i] = sar[i - 1] + af_value * (data['High'][i - 1] - sar[i - 1])
/var/folders/j_/0q2qhjf90s7gbt6hw8zb3_jc0000gn/T/ipykernel_44168/886000332.py:117: FutureWarning: Series.__setitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To set a value by position, use `ser.iloc[pos] = value`
  sar[i] = sar[i - 1] + af_value * (data['High'][i - 1] - sar[i - 1])
/var/folders/j_/0q2qhjf90s7gbt6hw8zb3_jc0000gn/T/ipykernel_44168/886000332.py:118: FutureWarning: Series.__getitem__ treating keys as positions is deprecate

| 14        | 0.3948    | 7.151     |


[*********************100%***********************]  1 of 1 completed
/var/folders/j_/0q2qhjf90s7gbt6hw8zb3_jc0000gn/T/ipykernel_44168/886000332.py:117: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  sar[i] = sar[i - 1] + af_value * (data['High'][i - 1] - sar[i - 1])
/var/folders/j_/0q2qhjf90s7gbt6hw8zb3_jc0000gn/T/ipykernel_44168/886000332.py:117: FutureWarning: Series.__setitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To set a value by position, use `ser.iloc[pos] = value`
  sar[i] = sar[i - 1] + af_value * (data['High'][i - 1] - sar[i - 1])
/var/folders/j_/0q2qhjf90s7gbt6hw8zb3_jc0000gn/T/ipykernel_44168/886000332.py:118: FutureWarning: Series.__getitem__ treating keys as positions is deprecate

| 15        | 0.3948    | 7.139     |


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [22:57:57] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


In [111]:
for i in best_years:
    print(f"Best training period: {i}")

Best training period: {'target': 0.39478956765018447, 'params': {'years': 7.282845382194266}}
Best training period: {'target': 0.3947896712503971, 'params': {'years': 7.139165354861602}}
Best training period: {'target': 0.39478965826347023, 'params': {'years': 7.151121264575306}}
Best training period: {'target': 0.39478976043305936, 'params': {'years': 7.187664318963276}}
